In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    PretrainedConfig, 
    PreTrainedTokenizerFast, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

import zipfile
import json
import torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


In [ ]:
# ==========================================
# 1. PHASE 3 CONFIGURATION & LABELS
# ==========================================

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
MODERN_DECODER_REPO = "A-240242/legal-modern-decoder-checkpoints" 
OUTPUT_DIR = "./phase3_encdec_lorma"
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"
# Official Phase 3 Labels
LABEL_LIST = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", 
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", 
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"
]
NUM_LABELS = len(LABEL_LIST)
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}


In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_flatten_rr_data(split_dir, files):
    """Flattens list-based document JSONs into sentence-level (text, label) pairs."""
    sentences = []
    labels = []
    
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/{split_dir}/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # Iterate over the list of document dicts directly
        for doc in data:
            doc_texts = doc["text"]
            doc_labels = doc["labels"]
            for text, label in zip(doc_texts, doc_labels):
                sentences.append(text)
                labels.append(LABEL2ID[label])
                
    return sentences, labels

print("Downloading and processing Phase 3 Data...")
train_texts, train_labels = load_and_flatten_rr_data("Train", ["CL_train.json", "IT_train.json"])
dev_texts, dev_labels = load_and_flatten_rr_data("Train", ["CL_dev.json", "IT_dev.json"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
# CRITICAL: For Causal sequence classification, right padding is standard, 
# but we must extract the hidden state of the *actual last token* before padding.
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_data(texts, labels):
    tokenized = tokenizer(texts, padding=False, truncation=True, max_length=512)
    tokenized["labels"] = labels
    return Dataset.from_dict(tokenized)

train_dataset = tokenize_data(train_texts, train_labels)
eval_dataset = tokenize_data(dev_texts, dev_labels)

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import hf_hub_download
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score
import math
# ==========================================
# 1. HELPER CLASSES (RoPE, RMSNorm, SwiGLU)
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float().to(device) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len=None):
        if seq_len > self.max_seq_len_cached:
            t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :])
        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class LormaAdapter(nn.Module):
    def __init__(self, hidden_size, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(hidden_size, rank, bias=False)
        self.B = nn.Linear(rank, hidden_size, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        return self.scaling * self.B(self.A(x))



# ==========================================
# 1. CONFIGURATION & LORMA ADAPTER
# ==========================================
class CustomLegalConfig(PretrainedConfig):
    model_type = "custom_legal_encdec"
    def __init__(self, vocab_size=32000, d_model=768, num_layers=6, num_heads=12, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff

class LormaAdapter(nn.Module):
    def __init__(self, d_model, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(d_model, rank, bias=False)
        self.B = nn.Linear(rank, d_model, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

# ==========================================
# 2. LORMA ENCODER & DECODER LAYERS
# ==========================================
class CustomLormaEncoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

    def forward(self, src, src_key_padding_mask=None):
        src2, _ = self.self_attn(src, src, src, key_padding_mask=src_key_padding_mask)
        src2 = src2 + self.lorma_attn(src2)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src2 = src2 + self.lorma_mlp(src2)
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class CustomLormaDecoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_self_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.multihead_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_cross_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.norm3 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, tgt, memory, tgt_mask=None, memory_key_padding_mask=None, tgt_key_padding_mask=None):
        tgt2, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt2 = tgt2 + self.lorma_self_attn(tgt2)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        tgt2, _ = self.multihead_attn(tgt, memory, memory, key_padding_mask=memory_key_padding_mask)
        tgt2 = tgt2 + self.lorma_cross_attn(tgt2)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt2 = tgt2 + self.lorma_mlp(tgt2)
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

# ==========================================
# 3. SEQUENCE CLASSIFICATION ARCHITECTURE
# ==========================================
class LormaEncDecForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, rank=128):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        self.d_model = config.d_model
        
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(1024, config.d_model)
        
        self.encoder = nn.ModuleList([CustomLormaEncoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        self.decoder = nn.ModuleList([CustomLormaDecoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        
        self.classifier = nn.Linear(config.d_model, num_labels)

    def _get_pos_ids(self, x):
        return torch.arange(x.size(1), device=x.device).unsqueeze(0).expand(x.size(0), -1)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device
        
        # Encoder Pass
        src_pos = self._get_pos_ids(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(src_pos)
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        for layer in self.encoder:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        memory = x
        
        # Decoder Pass 
        tgt_pos = self._get_pos_ids(input_ids)
        tgt = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(tgt_pos)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(input_ids.size(1), device=device)
        
        for layer in self.decoder:
            tgt = layer(
                tgt, 
                memory, 
                tgt_mask=tgt_mask, 
                memory_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=src_key_padding_mask
            )
            
        # Mean Pooling over the decoder output for sequence-level representation
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(tgt.size()).float()
            sum_embeddings = torch.sum(tgt * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled_logits = sum_embeddings / sum_mask
        else:
            pooled_logits = tgt.mean(dim=1)

        logits = self.classifier(pooled_logits)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)

In [11]:
# ==========================================
# 3. INITIALIZATION & WEIGHT TRANSFER
# ==========================================
import torch
import torch.nn as nn
from peft import get_peft_model, LoraConfig, TaskType
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.metrics import precision_recall_fscore_support
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing LORMA Encoder-Decoder Sequence Classifier...")
encdec_config = CustomLegalConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id
)
model = LormaEncDecForSequenceClassification(encdec_config, num_labels=NUM_LABELS, rank=128)

print("Loading and Remapping Phase 1 Weights...")
HF_MODEL_REPO = "A-240242/legal-encdec-checkpoints"
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(model_weight_path)

mapped_dict = {}
for key, val in state_dict.items():
    if key.startswith("encoder.layers."):
        mapped_dict[key.replace("encoder.layers.", "encoder.")] = val
    elif key.startswith("decoder.layers."):
        mapped_dict[key.replace("decoder.layers.", "decoder.")] = val
    else:
        mapped_dict[key] = val

model.load_state_dict(mapped_dict, strict=False)

# Freeze entire base model
for param in model.parameters(): 
    param.requires_grad = False

# Unfreeze LORMA matrices + classification head
for layer in model.encoder:
    for param in layer.lorma_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for layer in model.decoder:
    for param in layer.lorma_self_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_cross_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for param in model.classifier.parameters(): 
    param.requires_grad = True

# Enforce FP32 upscaling for trainable parameters to bypass AMP scaler issues
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Trainable Parameters: {trainable_params:,}")

Initializing LORMA Encoder-Decoder Sequence Classifier...
Loading and Remapping Phase 1 Weights...
Model ready. Trainable Parameters: 5,908,237


In [12]:
# ==========================================
# 5. METRICS & TRAINING EXECUTION
# ==========================================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

training_args_encdec = TrainingArguments(
    output_dir="./phase3_encdec_lorma",
    report_to="none",
    num_train_epochs=10, 
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,         
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=True,
    seed=42
)

encdec_trainer = Trainer(
    model=model,
    args=training_args_encdec,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\nCommencing Single-GPU LORMA Encoder-Decoder Sequence Classification Training...")
encdec_trainer.train()


Commencing Single-GPU LORMA Encoder-Decoder Sequence Classification Training...


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro Precision,Macro Recall,Macro F1
1,15.819736,3.984343,0.108875,0.166356,0.120563
2,14.036632,3.801616,0.235529,0.198514,0.169954
3,13.325817,3.677267,0.441680,0.265646,0.260179
4,12.531105,3.678803,0.340921,0.261248,0.251489
5,12.001079,3.706964,0.420763,0.244837,0.233588
6,11.382634,3.706081,0.397637,0.285693,0.283095
7,11.465618,3.674901,0.401842,0.292718,0.287794
8,11.266464,3.713460,0.414561,0.291807,0.288067
9,11.031388,3.735959,0.406753,0.316120,0.316857
10,10.763466,3.724215,0.383523,0.305526,0.305606


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/l

TrainOutput(global_step=2640, training_loss=12.518222670121627, metrics={'train_runtime': 4592.7378, 'train_samples_per_second': 36.677, 'train_steps_per_second': 0.575, 'total_flos': 0.0, 'train_loss': 12.518222670121627, 'epoch': 10.0})

In [13]:
# ==========================================
# 6. VALIDATION SET EVALUATION
# ==========================================
print("Running evaluation on the Development Set...")
eval_results = encdec_trainer.evaluate()
print(f"Validation Macro-F1: {eval_results['eval_macro_f1']:.4f}")
print(f"Validation Precision: {eval_results['eval_macro_precision']:.4f}")
print(f"Validation Recall: {eval_results['eval_macro_recall']:.4f}")

Running evaluation on the Development Set...


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro Precision,Macro Recall,Macro F1
10.763466,3.735959,10,0.406753,0.316120,0.316857


Validation Macro-F1: 0.3169
Validation Precision: 0.4068
Validation Recall: 0.3161


In [14]:
# ==========================================
# 7. LOAD TEST DATA
# ==========================================
def load_test_data(files):
    test_docs = []
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/Test/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            if isinstance(data, dict):
                for doc_id, content in data.items():
                    # If the content is just a list of sentences
                    if isinstance(content, list):
                        test_docs.append({"id": doc_id, "text": content})
                    # If it happens to be a dictionary
                    elif isinstance(content, dict):
                        if "id" not in content:
                            content["id"] = doc_id
                        test_docs.append(content)
                        
            elif isinstance(data, list):
                test_docs.extend(data)
                
    return test_docs

print("\nDownloading Test Data...")
test_docs = load_test_data(["CL_test.json", "IT_test.json"])
print(f"Successfully loaded {len(test_docs)} test documents.")

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Successfully loaded 10 test documents.


In [15]:
# ==========================================
# 8. TEST SET INFERENCE & SUBMISSION
# ==========================================
model .eval()
model .to(device)
predictions = {}

print("Generating Rhetorical Role Predictions...")
for doc in tqdm(test_docs, desc="Predicting Document Sentences"):
    doc_id = doc["id"]
    sentences = doc["text"]
    doc_predictions = []
    
    batch_size = 16
    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i : i + batch_size]
        
        inputs = tokenizer(
            batch_sentences, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            
        doc_predictions.extend([ID2LABEL[pred] for pred in preds])
        
    assert len(doc_predictions) == len(sentences), f"Count mismatch for {doc_id}"
    predictions[doc_id] = doc_predictions

Generating Rhetorical Role Predictions...


Predicting Document Sentences:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


In [16]:
# ==========================================
# 9. SAVE & ZIP FOR CODABENCH
# ==========================================
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nPhase 3 Predictions complete! Submission saved to: {SUBMISSION_ZIP}")


Phase 3 Predictions complete! Submission saved to: ./phase3_encdec_lorma/submission.zip
